## Running multiple circuits at once with CircuitBinding

This notebooks aims at presenting our equivalent in MPQP to qiskit's pubs and braket's ProgramSets.

The goal of this feature is to be able to send multiple circuits in one job and being able to retrieved the correct result with the run.

The term binding comes from the fact that you can "bind" a circuit with measurements (for example observables) and variables (in case of parametrized circuits) so that you can run the circuit(s) with different observables.

In [2]:
# First we setup here the circuits, values and observables to be used
from mpqp.core.circuit import BindingMode, CircuitBinding
from mpqp.execution.devices import (
    AvailableDevice,
    IBMDevice,
    AWSDevice,
)

from mpqp import (
    CNOT,
    BasisMeasure,
    ExpectationMeasure,
    H,
    IBMDevice,
    Language,
    Observable,
    QCircuit,
    U,
    pI,
    pZ,
    pX,
    run,
)
from sympy import Symbol
from mpqp.execution.runner import run
from mpqp.execution.devices import IBMDevice, AWSDevice

theta, phi, psi = Symbol('θ'), Symbol('phi'), Symbol('psi')
c1 = QCircuit([U(theta, phi, psi, 0)], label="c1")
c2 = QCircuit([H(0)], label="c2")
c3 = QCircuit([H(0), CNOT(0, 1)], label="c3")
c4 = QCircuit([H(0), H(1), CNOT(0, 1)], label="c4")

v1 = {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
v2 = {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
v3 = {'θ': 3.0, 'phi': 3.0, 'psi': 3.0}
v4 = {'θ': 4.0, 'phi': 4.0, 'psi': 4.0}

m1 = ExpectationMeasure(Observable(pI), label="Exp1", shots=2024)
m2 = ExpectationMeasure(Observable(pX @ pZ), label="Exp2", shots=2024)
m3 = BasisMeasure(label="b3", shots=2024)
m4 = None

m_I = ExpectationMeasure(Observable(pI), label="Exp_I", shots=2024)
m_Z = ExpectationMeasure(Observable(pZ), label="Exp_Z", shots=2024)

# Usecase 1: Running a list of circuit

The most basic case is the one where we just want to run circuits without any binding.

In [ ]:
# Example 1: Two state vector jobs
# Note: state vector jobs are not supported by braket's ProgramSets
binding = CircuitBinding([c3, c4])
print(run(binding, IBMDevice.AER_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      State vector: [0.70711, 0, 0, 0.70711]
      Probabilities: [0.5, 0, 0, 0.5]
      Number of qubits: 2
    

    Result: c4, IBMDevice, AER_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2
    



In [10]:
binding_with_basismeasure = CircuitBinding([c3 + QCircuit([m3]), c4 + QCircuit([m3])])
print(run(binding, IBMDevice.AER_SIMULATOR))

BatchResult: 2 results
    Result: c2, IBMDevice, AER_SIMULATOR
      State vector: [0.70711, 0.70711]
      Probabilities: [0.5, 0.5]
      Number of qubits: 1
    

    Result: c4, IBMDevice, AER_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2
    

